# 行为树树状图（缩进示意）

> 本图是 `SentryBehaviorTree.xml` 的结构化缩进视图，用于快速对齐 Groot / BT.CPP 的节点层级。

## MainSentryTree（主树）

```
rmuc_2026 (主树)
└─ Sequence
   ├─ SubTree: PerceptionAndBlackboard
   ├─ SubTree: InitOnce
   └─ WhileDoElse (IsMatchStage: game_progress=4, 0~420s)
      ├─ THEN: ReactiveSequence
      │  ├─ SubTree: CommandHub
      │  └─ ReactiveFallback (priority)
      │     ├─ SubTree: RespawnRecovery      ← 死亡/复活/回血一体化
      │     ├─ SubTree: CriticalSurvival
      │     ├─ SubTree: BaseDefense
      │     ├─ SubTree: EngageCombat
      │     ├─ SubTree: SustainAndEconomy
      │     ├─ SubTree: ObjectivePlanner
      │     └─ SubTree: PatrolAndScan
      └─ ELSE: ReactiveSequence
         ├─ RateController(1Hz) → SendGoal(Home)
         ├─ RmucRobotControl(stop=T, spin=F, fire=F)
         └─ RateController(2Hz) → SentryCmdMux(posture=Move)
```

**优先级说明**（ReactiveFallback 从上到下递减）：

| 优先级 | 子树 | 职责 |
|:---:|:---|:---|
| 0 (最高) | RespawnRecovery | 死亡等待 / 复活后导航补给区刷卡解虚弱并回血 |
| 1 | CriticalSurvival | 危急生存（HP↓ / 热量↑） |
| 2 | BaseDefense | 基地防御（基地受威胁时） |
| 3 | EngageCombat | 交战（有有效目标且允许战斗） |
| 4 | SustainAndEconomy | 后勤补给（回血 / 补弹） |
| 5 | ObjectivePlanner | 战略目标占领 |
| 6 (最低) | PatrolAndScan | 巡逻扫描 |

## 感知 & 初始化子树

### PerceptionAndBlackboard

```
PerceptionAndBlackboard
└─ Sequence
   ├─ RmucSubGameStatus      → {game_status}, {time.now_ms}
   ├─ RmucSubRobotStatus     → {robot_status}
   ├─ RmucSubRFIDStatus      → {rfid.status}
   ├─ RmucSubRobotPosition   → {pose.x/y/yaw}, {is_at_nav_goal}
   ├─ SubRadarTracks          → {radar.tracks}
   └─ ParseSentryBlackboard  → ~20 个派生黑板变量
```

> 📥 5 个独立话题订阅：`/game_status` · `/robot_status` · `/rfid_status` · `/robot_position` · `/radar/enemy_tracks`

### InitOnce

```
InitOnce
└─ Sequence
   ├─ InitSentryConfig → 话题名/坐标/阈值 → {cfg.*}
   └─ InitCmdState     → {cmd.state}, {cmd.allow_ammo_target}
```

### CommandHub

```
CommandHub
└─ Sequence
   ├─ DecidePosture    → {cmd.posture}
   ├─ DecideEconomyCmd → {cmd.allow_ammo_target}, {cmd.trig_remote_*}
   ├─ DecideRespawnCmd → {cmd.confirm_*}
   ├─ RateController(5Hz) → SentryCmdMux(0x0120)
   └─ KeepRunning
```

## 生存 & 防御子树

### RespawnRecovery（死亡 + 复活 + 刷卡回血 一体化）

```
RespawnRecovery
└─ Sequence
   ├─ DetectRespawnAndSetRecovery   ← 每 tick 跟踪 was_dead / 检测复活沿
   └─ Fallback
      │
      ├─ [A] IfDead_StopAndWait
      │  └─ Sequence
      │     ├─ RmucIsDead
      │     ├─ RmucRobotControl(stop=T, spin=F)
      │     └─ RmucNavControlCmd(cmd_type=3, 原地不动)
      │
      └─ [B] RecoveryMode (复活后强制回补给区)
         └─ Sequence
            ├─ IsRecoveryNeeded
            └─ Fallback (RecoveryFlow)
               │
               ├─ IfSupplyCard_Heal_Exit
               │  └─ Sequence
               │     ├─ RmucIsSupplyCardDetected
               │     ├─ RmucWaitAndHeal (等待回血到指定比例)
               │     └─ ClearRecoveryFlag
               │
               └─ GoSupply_ThenSearch
                  └─ Sequence
                     ├─ RmucNavControlCmd(cmd_type=1, 开始导航)
                     ├─ ReactiveFallback (NavUntilArrived)
                     │  ├─ RmucIsAtNavGoal
                     │  └─ ReactiveSequence
                     │     ├─ RateController(5Hz) → SendGoal(补给区)
                     │     └─ KeepRunning
                     ├─ RmucRobotControl(stop=T, spin=F)
                     ├─ RmucNavControlCmd(cmd_type=3, 原地不动)
                     ├─ InitSearchTimerIfNeeded
                     └─ ReactiveFallback (DetectOrMicroSearch)
                        ├─ RmucIsSupplyCardDetected
                        └─ RmucMicroSearchSupplyCard
```

> 状态机：存活(无 recovery) → FAILURE → 死亡 → SUCCESS(停车) → 复活沿 → recovery=true → 导航补给区 → 刷卡 → 回血 → 清 flag

### CriticalSurvival

```
CriticalSurvival
└─ ReactiveSequence
   ├─ IsCriticalState (HP < hp_critical 或 heat > heat_critical)
   ├─ CancelNavGoal
   ├─ RmucRobotControl(fire off)
   ├─ SelectSafeRetreatGoal → {nav.goal_x,y}
   ├─ RateController(1Hz) → SendGoal(Retreat)
   └─ KeepRunning
```

### BaseDefense

```
BaseDefense
└─ ReactiveSequence
   ├─ IsBaseThreatened
   ├─ RmucRobotControl(fire on)
   ├─ RateController(1Hz) → SendGoal(DefendAnchor)
   └─ SubTree: CombatLoop
```

## 战斗子树

### EngageCombat

```
EngageCombat
└─ ReactiveSequence
   ├─ HasValidTarget
   ├─ IsCombatAllowed
   ├─ RmucRobotControl(stop=T, spin=T, fire=T)
   └─ SubTree: CombatLoop
```

### CombatLoop

```
CombatLoop
└─ ReactiveSequence
   ├─ SelectBestTarget → {combat.best_target}
   ├─ AimAtTarget
   ├─ ReactiveFallback
   │  ├─ IsFireWindowOk
   │  └─ RmucRobotControl(fire off)
   ├─ FireBurst(burst/pause)
   └─ KeepRunning
```

## 后勤子树

### SustainAndEconomy

```
SustainAndEconomy
└─ ReactiveFallback
   ├─ SubTree: HealPlan
   └─ SubTree: AmmoPlan
```

### HealPlan

```
HealPlan
└─ ReactiveSequence
   ├─ RmucIsHPBelow(hp_threshold={cfg.hp_low})
   └─ ReactiveFallback
      ├─ (已在补给区) → Sequence
      │  ├─ IsZoneCardDetected(zone="SUPPLY")
      │  └─ HoldAndHeal (驻留至 hp ≥ hp_safe)
      └─ (前往补给区) → Sequence
         ├─ RateController(1Hz) → SendGoal(GoSupply)
         ├─ RmucRobotControl(fire off)
         └─ KeepRunning
```

### AmmoPlan

```
AmmoPlan
└─ ReactiveSequence
   ├─ IsAmmoBelow(ammo_low={cfg.ammo_low})
   └─ ReactiveFallback
      ├─ (已在补给区) → Sequence
      │  ├─ IsZoneCardDetected(zone="SUPPLY")
      │  └─ HoldForSupplyAmmoTick (等待弹量达标)
      └─ (前往最近补给站) → Sequence
         ├─ SelectNearestResupplyStation → {nav.goal_x,y}
         ├─ RateController(1Hz) → SendGoal(GoResupplyStation)
         ├─ RmucRobotControl(fire off)
         └─ KeepRunning
```

## 目标 & 巡逻子树

### ObjectivePlanner

```
ObjectivePlanner
└─ ReactiveSequence
   ├─ SelectObjective → {nav.objective}, {nav.goal_x,y}
   ├─ RateController(1Hz) → SendGoal({nav.objective})
   ├─ RmucRobotControl(fire off)
   ├─ ReactiveFallback
   │  ├─ IsAtGoal(arrive_radius)
   │  └─ KeepRunning
   └─ HoldObjective(hold_ms, 基地威胁/有敌人可提前结束)
```

### PatrolAndScan

```
PatrolAndScan
└─ Sequence
   ├─ RmucRobotControl(scan, fire off)
   ├─ WaypointPatrol → {nav.goal_x,y}
   ├─ RateController(1Hz) → SendGoal(Patrol)
   └─ KeepRunning
```